# Uber pickups — Préparation et exploration

**Premier notebook à exécuter**, de haut en bas, depuis le dossier `UBER_PICKUPS`,
avec l'environnement Python `certif`. Bibliothèques utilisées : pandas, GeoPandas et Plotly.

L'objectif est de repérer les lieux de prise en charge les plus fréquentés à New York.
Je reprends le chargement des fichiers, la jointure avec les zones de taxi et les cartes
des notebooks existants. Le notebook suivant compare KMeans et DBSCAN.

Les cellules précédées de **LIVRABLE** produisent un résultat à présenter.
Les autres cellules préparent ou vérifient l'analyse.

**Périmètre retenu : avril à septembre 2014**, car ces fichiers contiennent les coordonnées
réelles des prises en charge. Il s'agit d'une analyse historique, pas d'une prévision en temps réel.

## 1. Importer les bibliothèques

In [1]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

## 2. Charger les données

Je garde la liste explicite des fichiers et leur concaténation.
Aucune extraction ou suppression de fichiers n'est nécessaire : les CSV sont déjà disponibles.

In [2]:
files = [
    "data/uber-trip-data/uber-raw-data-apr14.csv",
    "data/uber-trip-data/uber-raw-data-may14.csv",
    "data/uber-trip-data/uber-raw-data-jun14.csv",
    "data/uber-trip-data/uber-raw-data-jul14.csv",
    "data/uber-trip-data/uber-raw-data-aug14.csv",
    "data/uber-trip-data/uber-raw-data-sep14.csv",
]

raw_2014 = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
raw_2014.columns = raw_2014.columns.str.lower()
raw_2014["date/time"] = pd.to_datetime(raw_2014["date/time"], format="%m/%d/%Y %H:%M:%S")

print("Rows / columns:", raw_2014.shape)
print("First pickup:", raw_2014["date/time"].min())
print("Last pickup:", raw_2014["date/time"].max())
display(raw_2014.head())

Rows / columns: (4534327, 4)
First pickup: 2014-04-01 00:00:00
Last pickup: 2014-09-30 22:59:00


,date/time,lat,lon,base
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512


In [3]:
print("Missing values:")
display(raw_2014.isna().sum())
print("Identical rows, kept:", raw_2014.duplicated().sum())
display(raw_2014.describe(include="all"))

Missing values:


date/time    0
lat          0
lon          0
base         0
dtype: int64

Identical rows, kept: 82581


,date/time,lat,lon,base
count,4534327,4.534327e+06,4.534327e+06,4534327
unique,NaN,NaN,NaN,5
top,NaN,NaN,NaN,B02617
freq,NaN,NaN,NaN,1458853
mean,2014-07-11 18:50:50.578151424,4.073926e+01,-7.397302e+01,NaN
min,2014-04-01 00:00:00,3.965690e+01,-7.492900e+01,NaN
25%,2014-05-28 15:18:00,4.072110e+01,-7.399650e+01,NaN
50%,2014-07-17 14:45:00,4.074220e+01,-7.398340e+01,NaN
75%,2014-08-27 21:55:00,4.076100e+01,-7.396530e+01,NaN
max,2014-09-30 22:59:00,4.211660e+01,-7.206660e+01,NaN


Les coordonnées et les dates ne présentent pas de valeur manquante dans les fichiers étudiés.
Je conserve les lignes identiques : sans identifiant de course, plusieurs prises en charge
au même endroit et à la même minute ne sont pas nécessairement des erreurs.

### Pourquoi ne pas réutiliser `sample.csv` ?

Dans `data_cleaning.ipynb`, les coordonnées médianes de chaque base de 2014 étaient attribuées
aux trajets de 2015. Cela réduisait artificiellement les positions à quelques points.
Une base Uber n'est pas un lieu de prise en charge. Je repars donc des coordonnées GPS de 2014.

In [4]:
raw_2015_preview = pd.read_csv("data/uber-raw-data-janjune-15.csv", nrows=5)
display(raw_2015_preview)

,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,B02617,2015-05-17 09:47:00,B02617,141
1,B02617,2015-05-17 09:47:00,B02617,65
2,B02617,2015-05-17 09:47:00,B02617,100
3,B02617,2015-05-17 09:47:00,B02774,80
4,B02617,2015-05-17 09:47:00,B02617,90


Le fichier de 2015 contient `locationID`, mais pas de latitude ni de longitude.
Le point représentatif d'une zone de taxi serait une approximation commune à tous ses trajets.
Je ne le mélange pas aux positions GPS de 2014 et je ne fusionne pas les trajets des deux années.

## 3. Garder les prises en charge de New York

Je reprends ta jointure spatiale : chaque point GPS est associé au polygone de sa zone de taxi.
Je garde Manhattan, Brooklyn, Queens, le Bronx et Staten Island. Newark est exclu.
Les latitudes et longitudes restent celles des trajets.

Le prédicat `within` exclut les points non situés à l'intérieur des polygones retenus.
Cela peut aussi écarter des points proches d'une limite ou d'une rive.

In [5]:
zones_official = gpd.read_file("data/taxi_zones/taxi_zones/taxi_zones.shp")
zones_official = zones_official[["LocationID", "zone", "borough", "geometry"]]
zones_official = zones_official.to_crs("EPSG:4326")

boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
zones_official = zones_official[zones_official["borough"].isin(boroughs)].copy()

display(zones_official[["borough", "zone"]].head())

,borough,zone
1,Queens,Jamaica Bay
2,Bronx,Allerton/Pelham Gardens
3,Manhattan,Alphabet City
4,Staten Island,Arden Heights
5,Staten Island,Arrochar/Fort Wadsworth


In [6]:
loc_2014 = gpd.GeoDataFrame(
    raw_2014,
    geometry=gpd.points_from_xy(raw_2014["lon"], raw_2014["lat"]),
    crs="EPSG:4326",
)

dataset_2014 = gpd.sjoin(loc_2014, zones_official, how="inner", predicate="within")
dataset_2014 = dataset_2014.drop(columns=["index_right"]).reset_index(drop=True)

print("Initial pickups:", len(raw_2014))
print("Pickups kept in NYC polygons:", len(dataset_2014))
print("Pickups excluded:", len(raw_2014) - len(dataset_2014))
display(dataset_2014[["date/time", "lat", "lon", "borough", "zone"]].head())

Initial pickups: 4534327
Pickups kept in NYC polygons: 4411894
Pickups excluded: 122433


,date/time,lat,lon,borough,zone
0,2014-04-01 00:11:00,40.7690,-73.9549,Manhattan,Lenox Hill East
1,2014-04-01 00:21:00,40.7316,-73.9873,Manhattan,East Village
2,2014-04-01 00:28:00,40.7588,-73.9776,Manhattan,Midtown Center
3,2014-04-01 00:33:00,40.7594,-73.9722,Manhattan,Midtown East
4,2014-04-01 00:39:00,40.7223,-73.9887,Manhattan,Lower East Side


## 4. Commencer par un jour et une heure

In [7]:
begin = pd.Timestamp("2014-04-01 00:00:00")
end = begin + pd.Timedelta(hours=1)

t1 = dataset_2014[
    (dataset_2014["date/time"] >= begin)
    & (dataset_2014["date/time"] < end)
].copy()

print("Pickups in the selected hour:", len(t1))

fig_one_hour = px.scatter_map(
    t1,
    lat="lat",
    lon="lon",
    color="borough",
    hover_name="zone",
    hover_data={"date/time": True, "lat": ":.4f", "lon": ":.4f"},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge du 1er avril 2014, de 0 h à 1 h",
)
fig_one_hour.show()

Pickups in the selected hour: 132


## 5. Ajouter le jour de semaine et l'heure

Chaque type de jour n'apparaît pas nécessairement autant de fois dans la période.
Je divise donc son volume total par le nombre de dates correspondantes pour comparer
des **prises en charge moyennes par journée**.

In [8]:
weekday_names = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
weekday_labels = dict(enumerate(weekday_names))

dataset_2014["date"] = dataset_2014["date/time"].dt.date
dataset_2014["weekday_number"] = dataset_2014["date/time"].dt.dayofweek
dataset_2014["weekday"] = dataset_2014["weekday_number"].map(weekday_labels)
dataset_2014["hour"] = dataset_2014["date/time"].dt.hour

daily_counts = dataset_2014.groupby(["weekday_number", "weekday"]).agg(
    pickups=("date/time", "size"),
    days=("date", "nunique"),
).reset_index().sort_values("weekday_number")

daily_counts["pickups_per_day"] = daily_counts["pickups"] / daily_counts["days"]

hourly_counts = dataset_2014.groupby(["weekday_number", "weekday", "hour"]).size()
hourly_counts = hourly_counts.reset_index(name="pickups")
hourly_counts = hourly_counts.merge(daily_counts[["weekday", "days"]], on="weekday", validate="many_to_one")
hourly_counts["pickups_per_day"] = hourly_counts["pickups"] / hourly_counts["days"]

### LIVRABLE — Activité par jour de semaine

**La cellule suivante produit** le tableau et le graphique des volumes moyens par jour.
Les volumes utilisent toutes les prises en charge conservées, avant échantillonnage.

In [9]:
display(daily_counts.round(1))

fig_activity = px.bar(
    daily_counts,
    x="weekday",
    y="pickups_per_day",
    category_orders={"weekday": weekday_names},
    labels={"weekday": "Jour", "pickups_per_day": "Prises en charge moyennes par journée"},
    title="Activité moyenne à New York par jour de semaine",
)
fig_activity.show()

,weekday_number,weekday,pickups,days,pickups_per_day
0,0,Lundi,525858,26,20225.3
1,1,Mardi,648767,27,24028.4
2,2,Mercredi,681292,26,26203.5
3,3,Jeudi,737424,26,28362.5
4,4,Vendredi,722211,26,27777.3
5,5,Samedi,625297,26,24049.9
6,6,Dimanche,471045,26,18117.1


### LIVRABLE — Activité par jour et par heure

**La cellule suivante produit** une carte de chaleur temporelle.
Une case correspond au volume moyen observé à cette heure pour ce jour de semaine.

In [10]:
hourly_matrix = hourly_counts.pivot(index="weekday", columns="hour", values="pickups_per_day")
hourly_matrix = hourly_matrix.reindex(weekday_names)

fig_hours = px.imshow(
    hourly_matrix,
    labels={"x": "Heure", "y": "Jour", "color": "Prises en charge moyennes"},
    aspect="auto",
    title="Quand ont lieu les prises en charge ?",
)
fig_hours.update_xaxes(dtick=1)
fig_hours.show()

## 6. Préparer un échantillon pour les algorithmes

Je tire au hasard **5 000 trajets par jour de semaine**, avec `random_state=42` pour retrouver
le même échantillon à chaque exécution. Les deux algorithmes utiliseront exactement ces mêmes points.
Les jours sont analysés séparément ; cet échantillon équilibré ne sert pas à comparer leurs volumes totaux.

Je projette ensuite les coordonnées dans le système **UTM 18N (`EPSG:32618`)** et les exprime
en kilomètres. KMeans et DBSCAN pourront utiliser la même distance euclidienne.
Un `StandardScaler` n'est pas nécessaire : les deux axes ont déjà la même unité.

Le second notebook refait cette préparation à partir des mêmes fichiers bruts. Aucun fichier intermédiaire n’est écrit.

In [11]:
sample = dataset_2014.groupby("weekday_number", group_keys=False).sample(n=5000, random_state=42)
sample = sample.copy()

sample_projected = sample.to_crs("EPSG:32618")
sample["x_km"] = sample_projected.geometry.x / 1000
sample["y_km"] = sample_projected.geometry.y / 1000

sample = pd.DataFrame(sample.drop(columns=["geometry", "date"]))

print("Sample rows:", len(sample))
display(sample.groupby("weekday_number").size().rename("sample_pickups"))
display(sample.head())

Sample rows: 35000


weekday_number
0    5000
1    5000
2    5000
3    5000
4    5000
5    5000
6    5000
Name: sample_pickups, dtype: int64

,date/time,lat,lon,base,LocationID,zone,borough,weekday_number,weekday,hour,x_km,y_km
1397427,2014-06-23 07:52:00,40.7226,-73.9982,B02598,144,Little Italy/NoLiTa,Manhattan,0,Lundi,7,584.605809,4508.446348
236975,2014-04-07 15:10:00,40.7623,-73.9680,B02617,237,Upper East Side South,Manhattan,0,Lundi,15,587.104542,4512.882871
3617892,2014-09-22 13:24:00,40.7591,-73.9774,B02598,161,Midtown Center,Manhattan,0,Lundi,13,586.315276,4512.518357
993518,2014-05-05 14:48:00,40.7245,-73.9975,B02682,144,Little Italy/NoLiTa,Manhattan,0,Lundi,14,584.662520,4508.657936
4400755,2014-09-29 13:22:00,40.8225,-73.9488,B02764,116,Hamilton Heights,Manhattan,0,Lundi,13,588.645009,4519.584805


## Ce que je retiens

Les coordonnées des prises en charge sont conservées. Les noms de zones viennent uniquement
de la jointure géographique. L'activité est décrite sur toutes les données retenues ;
le clustering sera réalisé sur un échantillon pour garder un temps de calcul raisonnable.

Les données sont des courses réalisées : elles ne mesurent ni les demandes annulées,
ni le nombre de chauffeurs disponibles. Elles permettent de repérer une fréquentation historique.